In [101]:
from generate_utils import load_GraphModel, load_BiLSTMModel
import torch
import numpy as np
import pickle
from tqdm import tqdm
from GridMLM_tokenizers import CSGridMLMTokenizer

In [102]:
tokenizer = CSGridMLMTokenizer(
    fixed_length=80,
    quantization='4th',
    intertwine_bar_info=True,
    trim_start=False,
    use_pc_roll=True,
    use_full_range_melody=False
)

In [103]:
device_name = 'cuda:0'
device = torch.device(device_name)

graph_model_path = 'saved_models/graph/graph_model.pt'
bilstm_model_path = 'saved_models/bilstm/bilstm_model.pt'

In [104]:
graph_model = load_GraphModel(graph_model_path, device)
bilstm_model = load_BiLSTMModel(bilstm_model_path, device)

In [105]:
datasets = {
    # 'gjt': {'path': 'data/gjt_test.pkl', 'dataset': None},
    # 'hook': {'path': 'data/hook_test.pkl', 'dataset': None},
    'nott': {'path': 'data/nott_test.pkl', 'dataset': None},
    # 'wiki': {'path': 'data/wiki_test.pkl', 'dataset': None}
}

In [106]:
for k, v in datasets.items():
    print(f'loading {k}')
    with open(v['path'], 'rb') as f:
        d = pickle.load(f)
    v['dataset'] = d

loading nott


In [107]:
# print(datasets['gjt']['dataset'][0]['graph_ready_object'].bar_objects)

In [108]:
# print(datasets['gjt']['dataset'][0]['graph_ready_object'].segment_graph)

In [109]:
# datasets['gjt']['dataset'][0]['graph_ready_object'].make_graph_of_segment(0,2)
# datasets['gjt']['dataset'][0]['graph_ready_object'].make_bilstm_seq_of_segment(0,2)

In [110]:
# print(datasets['gjt']['dataset'][0]['graph_ready_object'].__dict__)

In [111]:
# print(datasets['gjt']['dataset'][0]['graph_ready_object'].segment_bilstm.shape)

In [112]:
# datasets['gjt']['dataset'][0]['graph_ready_object'].bar_objects

In [113]:
graph_embeddings = []
bilstm_embeddings = []
metadata = []

for k,v in datasets.items():
    for i, d in tqdm(enumerate(v['dataset'])):
        g = d['graph_ready_object']
        bar_objects = g.bar_objects
        for bar_start in range(len(bar_objects)-2):
            bar_end = bar_start + 2
            # graph
            g.make_graph_of_segment(bar_start, bar_end)
            z_graph = graph_model(g.segment_graph.to(device))
            graph_embeddings.append(
                z_graph.detach().cpu().numpy()
            )
            # bilstm
            g.make_bilstm_seq_of_segment(bar_start, bar_end)
            segment_bilstm = g.segment_bilstm.unsqueeze(0)
            lengths = torch.tensor(segment_bilstm.shape[1], dtype=int).unsqueeze(0)
            z_bilstm = bilstm_model(segment_bilstm.to(device), lengths.to(device))
            bilstm_embeddings.append(
                z_bilstm.detach().cpu().numpy()
            )
            # make string of chord symbols
            chords_str = ''
            for bar_idx in range(bar_start, bar_end):
                for chord in bar_objects[bar_idx].chord_objects:
                    chords_str += tokenizer.ids_to_tokens[chord.chord_id] + ' '
            metadata.append({
                'dataset': k,
                'piece_idx': i,
                'bar_start': bar_start,
                'bar_end': bar_end,
                'chords': chords_str
            })

graph_embeddings = np.vstack(graph_embeddings)
bilstm_embeddings = np.vstack(bilstm_embeddings)

45it [00:00, 74.72it/s]


In [114]:
print(graph_embeddings.shape)
print(bilstm_embeddings.shape)

(664, 128)
(664, 128)


In [115]:
print(metadata[100])

{'dataset': 'nott', 'piece_idx': 6, 'bar_start': 9, 'bar_end': 11, 'chords': 'A:min E:7 '}


In [116]:
from sklearn.decomposition import PCA

In [117]:
pca = PCA(n_components=2)

In [118]:
y_graph = pca.fit_transform(graph_embeddings)
y_bilstm = pca.fit_transform(bilstm_embeddings)

In [119]:
print(y_graph.shape, y_bilstm.shape)

(664, 2) (664, 2)


In [120]:
import plotly.express as px
import pandas as pd

In [121]:
chords_list = [metadata[i]['chords'] for i in range(len(y_graph))]
df = pd.DataFrame({
    'x': y_graph[:,0],
    'y': y_graph[:,1],
    'chords': chords_list
})

fig = px.scatter(
    data_frame=df,
    x='x',
    y='y',
    hover_data=['chords']
)
fig.show()

In [122]:
chords_list = [metadata[i]['chords'] for i in range(len(y_graph))]
df = pd.DataFrame({
    'x': y_bilstm[:,0],
    'y': y_bilstm[:,1],
    'chords': chords_list
})

fig = px.scatter(
    data_frame=df,
    x='x',
    y='y',
    hover_data=['chords']
)
fig.show()